# 02 · Silver — temiz, tipli ve tekilleştirilmiş maliyet kayıtları

1. **En güncel export run'ını seç:** Günlük *MonthToDate* export her gün ayın tamamını yeniden yazar.
   Eski run'lar da okunursa maliyet **iki kez** sayılır. Her fatura dönemi için en yeni run tutulur.
2. **FOCUS → sabit sözleşme:** Kolon adları büyük/küçük harf duyarsız eşlenir; FOCUS varyantları
   (`x_ResourceGroupName` / `ResourceGroupName` …) birleştirilir.
3. **Etiketler (tags):** JSON → anahtarları küçük harfe çevrilmiş map → `environment`, `costCenter`, `project`.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StringType, StructType

spark.conf.set("spark.sql.session.timeZone", "UTC")   # FOCUS timestamps are UTC

bronze = spark.read.format("delta").load("Tables/dbo/bronze_costs")


def pick(df, *names, dtype="string"):
    """Case-insensitive coalesce over FOCUS column-name variants; NULL if none exist."""
    cols = {c.lower(): c for c in df.columns}
    exprs = [F.col(f"`{cols[n.lower()]}`").cast(dtype) for n in names if n.lower() in cols]
    return F.coalesce(*exprs) if exprs else F.lit(None).cast(dtype)


def tags_json(df):
    field = next((f for f in df.schema.fields if f.name.lower() == "tags"), None)
    if field is None:
        return F.lit(None).cast("string")
    if isinstance(field.dataType, (MapType, StructType, ArrayType)):
        return F.to_json(F.col(f"`{field.name}`"))
    value = F.trim(F.col(f"`{field.name}`").cast("string"))
    # Older exports omit the outer braces: "env": "prod"
    return F.when(value.startswith("{"), value).otherwise(F.concat(F.lit("{"), value, F.lit("}")))

## 1. Fatura dönemi başına en güncel export run'ı

In [ ]:
with_run = (
    bronze
    .withColumn("_billing_period", F.regexp_extract("_source_file", r"/(\d{8}-\d{8})/", 1))
    .withColumn("_export_run", F.regexp_extract("_source_file", r"/\d{8}-\d{8}/([^/]+)/", 1))
)

latest_runs = (
    with_run.groupBy("_billing_period", "_export_run")
    .agg(F.max("_file_modification_time").alias("_run_time"))
    .withColumn("_rank", F.row_number().over(Window.partitionBy("_billing_period").orderBy(F.desc("_run_time"))))
    .filter("_rank = 1")
    .select("_billing_period", "_export_run")
)

current = with_run.join(latest_runs, ["_billing_period", "_export_run"], "inner")
print(f"Bronze: {bronze.count():,} satır  ->  en güncel run'lar: {current.count():,} satır")

## 2. FOCUS kolonlarını Silver sözleşmesine dönüştür

In [ ]:
resource_id = F.lower(pick(current, "ResourceId"))
rg_from_id = F.regexp_extract(resource_id, r"/resourcegroups/([^/]+)", 1)
tags = tags_json(current)
tag_map = F.transform_keys(F.from_json(tags, MapType(StringType(), StringType())), lambda k, v: F.lower(k))

silver = (
    current.select(
        F.to_date(pick(current, "ChargePeriodStart", dtype="timestamp")).alias("charge_date"),
        F.to_date(pick(current, "BillingPeriodStart", dtype="timestamp")).alias("billing_period_start"),
        pick(current, "BillingAccountName").alias("billing_account_name"),
        F.regexp_replace(F.lower(pick(current, "SubAccountId")), "^/subscriptions/", "").alias("subscription_id"),
        pick(current, "SubAccountName").alias("subscription_name"),
        F.coalesce(
            F.lower(pick(current, "x_ResourceGroupName", "ResourceGroupName", "ResourceGroup")),
            F.when(rg_from_id != "", rg_from_id),
        ).alias("resource_group"),
        resource_id.alias("resource_id"),
        pick(current, "ResourceName").alias("resource_name"),
        pick(current, "x_ResourceType", "ResourceType").alias("resource_type"),
        F.coalesce(pick(current, "ServiceName"), F.lit("Unknown")).alias("service_name"),
        F.coalesce(pick(current, "ServiceCategory"), F.lit("Other")).alias("service_category"),
        pick(current, "x_SkuMeterCategory").alias("meter_category"),
        pick(current, "x_SkuMeterName").alias("meter_name"),
        pick(current, "RegionName", "RegionId").alias("region"),
        pick(current, "ChargeCategory").alias("charge_category"),
        pick(current, "ChargeDescription").alias("charge_description"),
        pick(current, "PricingCategory").alias("pricing_category"),
        pick(current, "BillingCurrency").alias("billing_currency"),
        pick(current, "BilledCost", dtype="double").alias("billed_cost"),
        pick(current, "EffectiveCost", "BilledCost", dtype="double").alias("effective_cost"),
        pick(current, "ListCost", "EffectiveCost", dtype="double").alias("list_cost"),
        pick(current, "ConsumedQuantity", dtype="double").alias("consumed_quantity"),
        pick(current, "ConsumedUnit").alias("consumed_unit"),
        tags.alias("tags"),
        F.coalesce(tag_map["environment"], tag_map["env"]).alias("tag_environment"),
        F.coalesce(tag_map["costcenter"], tag_map["cost-center"], tag_map["cost_center"]).alias("tag_cost_center"),
        F.coalesce(tag_map["project"], tag_map["application"], tag_map["app"]).alias("tag_project"),
        "_billing_period",
        "_export_run",
        "_source_file",
    )
    # No row-level dedup: identical FOCUS line items are legitimate; duplicates come only from old export runs.
    .filter(F.col("charge_date").isNotNull() & F.col("effective_cost").isNotNull())
    .withColumn("_silver_timestamp", F.current_timestamp())
)

## 3. Kontrol ve yazma

In [ ]:
display(
    silver.groupBy("service_name", "billing_currency")
    .agg(F.round(F.sum("effective_cost"), 2).alias("effective_cost"), F.count("*").alias("line_items"))
    .orderBy(F.desc("effective_cost"))
)

(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/dbo/silver_costs")
)
print(f"silver_costs yazıldı: {spark.read.format('delta').load('Tables/dbo/silver_costs').count():,} satır")